# Atención eficiente y KV cache en PyTorch

Notebook opcional del **módulo 8** del curso (en la app: `uv run dev` y la ruta `#/modulo/variantes-atencion`). Usa modelos reales; el módulo interactivo explica la teoría con NumPy.

> **Requisitos:** desde la raíz del repositorio, `uv sync --group notebooks` y abre este notebook con `uv run --group notebooks jupyter lab`. La primera ejecución descarga los modelos de Hugging Face.

In [ ]:
import time

import torch
import torch.nn.functional as F

B, H, N, D = 1, 8, 2048, 64
q, k, v = (torch.randn(B, H, N, D) for _ in range(3))

In [ ]:
def manual(q, k, v):
    s = q @ k.transpose(-2, -1) / D ** 0.5
    return s.softmax(-1) @ v

for nombre, f in [("manual", manual), ("SDPA", F.scaled_dot_product_attention)]:
    t = time.perf_counter()
    for _ in range(5):
        y = f(q, k, v)
    print(nombre, round((time.perf_counter() - t) / 5 * 1000, 1), "ms")
print("¿iguales?", torch.allclose(manual(q, k, v), F.scaled_dot_product_attention(q, k, v), atol=1e-4))

El efecto de la KV cache al generar con un modelo real.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

tok = AutoTokenizer.from_pretrained("gpt2")
modelo = AutoModelForCausalLM.from_pretrained("gpt2")
x = tok("The key-value cache", return_tensors="pt")
for cache in [True, False]:
    t = time.perf_counter()
    modelo.generate(**x, max_new_tokens=100, do_sample=False, use_cache=cache, pad_token_id=tok.eos_token_id)
    print("use_cache =", cache, round(time.perf_counter() - t, 2), "s")